In [1]:
# shop_masterを読み込みデータを表示します。
import pandas as pd

shop_master = pd.read_csv("../data/shop_master.csv")
shop_master.head()

,shop_id,zip,shop_name,address,Latitude,Longitude
0,S000001,053-8722,北海道 苫小牧出張所,北海道苫小牧市旭町4-5-6,42°38′03″N,141°36′20″E
1,S000002,154-8504,東京 成城支店,東京都世田谷区世田谷4-21-27,35°38′48″N,139°39′12″E
2,S000003,105-8511,東京 白金台支店,東京都港区芝公園1-5-25,35°38′48″N,139°43′22″E
3,S000004,400-8585,山梨県 甲府支店,山梨県甲府市丸の内1-18-1,35°39′44″N,138°34′06″E
4,S000005,75001,フランス パリ支店,フランス パリ,48°51′24″N,2°21′3″E


In [2]:
# データ型の確認をしてみましょう
print(shop_master.dtypes)

shop_id      object
zip          object
shop_name    object
address      object
Latitude     object
Longitude    object
dtype: object


In [3]:
# LatitudeとLongitudeを十進法に変換します。
def convert_dms_to_dd(dms_str):
    """
    度分秒(DMS)形式の文字列を十進法(DD)に変換する関数、dms_strを用います。
    度分秒形式の文字列（例: '35°41′23″N'）を渡すと、十進法で表現された緯度または経度に変換し、変換できない場合は元の文字列を返します。"""

    try:
        parts = dms_str.split("°")
        degrees = float(parts[0])
        minutes_part = parts[1].split("′")
        minutes = float(minutes_part[0])
        seconds_part = minutes_part[1].split("″")
        seconds = float(seconds_part[0])

        direction = seconds_part[1]  # N, S, E, W

        dd = degrees + (minutes / 60) + (seconds / 3600)
        if direction in ("S", "W"):
            dd *= -1
        return dd

    except (IndexError, ValueError):
        return dms_str  # 変換できない場合は元の値を返す


# 緯度と経度の列名を確認し、必要があれば修正
# shop_master.columnsで列名を確認できます。
# 例：列名が'Latitude'と'Longitude'の場合
latitude_column = "Latitude"  # 'latitude'を実際の列名に置き換える
longitude_column = "Longitude"  # 'longitude'を実際の列名に置き換える

# 緯度と経度の列を十進法に変換
shop_master["latitude_dd"] = shop_master[latitude_column].apply(convert_dms_to_dd)
shop_master["longitude_dd"] = shop_master[longitude_column].apply(convert_dms_to_dd)

shop_master.head()

,shop_id,zip,shop_name,address,Latitude,Longitude,latitude_dd,longitude_dd
0,S000001,053-8722,北海道 苫小牧出張所,北海道苫小牧市旭町4-5-6,42°38′03″N,141°36′20″E,42.634167,141.605556
1,S000002,154-8504,東京 成城支店,東京都世田谷区世田谷4-21-27,35°38′48″N,139°39′12″E,35.646667,139.653333
2,S000003,105-8511,東京 白金台支店,東京都港区芝公園1-5-25,35°38′48″N,139°43′22″E,35.646667,139.722778
3,S000004,400-8585,山梨県 甲府支店,山梨県甲府市丸の内1-18-1,35°39′44″N,138°34′06″E,35.662222,138.568333
4,S000005,75001,フランス パリ支店,フランス パリ,48°51′24″N,2°21′3″E,48.856667,2.350833


In [ ]:
# shop_masterのlatitude_ddとlongituide_ddをUTM形式に変換
# はじめにUTMライブラリをインストールし、呼び出し
# !pip install utm
import utm

# 緯度・軽度のDD形式をUTMに変更します
def convert_dd_to_utm(latitude, longitude):
    """Converts latitude and longitude (decimal degrees) to UTM coordinates."""
    try:
        utm_coords = utm.from_latlon(latitude, longitude)
        return utm_coords
    except Exception as e:
        print(f"Error converting {latitude}, {longitude} to UTM: {e}")
        return None

# Apply the conversion function to create new columns
shop_master[['utm_easting', 'utm_northing', 'utm_zone_number', 'utm_zone_letter']] = shop_master.apply(
    lambda row: pd.Series(convert_dd_to_utm(row['latitude_dd'], row['longitude_dd'])), axis=1)


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Error converting 31°13′N, 121°28′E to UTM: ufunc 'minimum' did not contain a loop with signature matching types (dtype('<U7'), dtype('<U7')) -> None
Error converting 23°33′S, 46°38′W to UTM: ufunc 'minimum' did not contain a loop with signature matching types (dtype('<U7'), dtype('<U7')) -> None
Error converting 19°26′N, 99°8′W to UTM: ufunc 'minimum' did not contain a loop with signature matching types (dtype('<U7'), dtype('<U7')) -> None
Error converting 30°3′N, 31°14′E to UTM: ufunc 'minimum' did not contain a loop with signature matching types (dtype('<U6'), dtype('<U6')) -> None
Error converting 39°55′N, 116°23′E to UTM: ufunc 'minimum' did not contain a loop with signature matching types (dtype('<U7'), dtype('<U7')) -> None
Error converting 23°42′N, 90°22′E to UTM: ufunc 'minimum' did not contain a loop with signature matching types (dtype('<U7'), dtype('<U7')) -> None
E

In [5]:
# データフレームを出力し、確認
shop_master.head()

,shop_id,zip,shop_name,address,Latitude,Longitude,latitude_dd,longitude_dd,utm_easting,utm_northing,utm_zone_number,utm_zone_letter
0,S000001,053-8722,北海道 苫小牧出張所,北海道苫小牧市旭町4-5-6,42°38′03″N,141°36′20″E,42.634167,141.605556,549649.725382,4.720369e+06,54.0,T
1,S000002,154-8504,東京 成城支店,東京都世田谷区世田谷4-21-27,35°38′48″N,139°39′12″E,35.646667,139.653333,378085.405783,3.945595e+06,54.0,S
2,S000003,105-8511,東京 白金台支店,東京都港区芝公園1-5-25,35°38′48″N,139°43′22″E,35.646667,139.722778,384372.601833,3.945511e+06,54.0,S
3,S000004,400-8585,山梨県 甲府支店,山梨県甲府市丸の内1-18-1,35°39′44″N,138°34′06″E,35.662222,138.568333,279887.560712,3.949209e+06,54.0,S
4,S000005,75001,フランス パリ支店,フランス パリ,48°51′24″N,2°21′3″E,48.856667,2.350833,452382.348685,5.411725e+06,31.0,U


In [ ]:
import requests


def get_address_google(latitude, longitude, api_key):
    url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latitude},{longitude}&key={api_key}&language=ja"

    response = requests.get(url)
    data = response.json()

    if data["status"] == "OK":
        return data["results"][0]["formatted_address"]
    else:
        return f"Error: {data['status']}"


# 使用例（APIキーは取得して設定する必要があります）
latitude = 35.701778770219356
longitude = 139.74196324721953

address = get_address_google(latitude, longitude, api_key)
print("住所:", address)

In [ ]:
# shop_masterからlatitude_dd	longitude_ddを取得
# 緯度と経度のカラムのみ抽出
latitude_longitude_data = shop_master[["latitude_dd", "longitude_dd"]]
latitude_longitude_data

In [ ]:
# 緯度経度のリストを読み込んで、住所に変換
# 緯度と経度のカラムのみ抽出
latitude_longitude_data = shop_master[["latitude_dd", "longitude_dd"]]
# 住所情報を格納するリストを定義します
addresses = []

In [ ]:
# 各緯度経度に対して住所を取得
for index, row in latitude_longitude_data.iterrows():
    latitude = row["latitude_dd"]
    longitude = row["longitude_dd"]

    # 緯度経度が数値でない場合はスキップ
    if not isinstance(latitude, (int, float)) or not isinstance(
        longitude, (int, float)
    ):
        addresses.append("Invalid latitude or longitude")
        continue

    address = get_address_google(latitude, longitude, api_key)
    addresses.append(address)

# 住所情報のリストをデータフレームに追加
shop_master["address"] = addresses

# 結果のデータフレームを表示
shop_master

In [ ]:
# 住所から緯度経度を出力するサンプルコード


def get_lat_lon_google(address, api_key):
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={api_key}&language=ja"

    response = requests.get(url)
    data = response.json()

    if data["status"] == "OK":
        location = data["results"][0]["geometry"]["location"]
        return location["lat"], location["lng"]
    else:
        return None, f"Error: {data['status']}"


# 使用例（APIキーは取得して設定する必要があります）
address = "東京都渋谷区代々木2−30−4"

latitude, longitude = get_lat_lon_google(address, api_key)

if latitude is not None:
    print("緯度:", latitude)
    print("経度:", longitude)
else:
    print("エラー:", longitude)